### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output. 

### Pydantic

Pydantic models provide the richest feature set with field validation, description and nested structure.

In [5]:
from dotenv import load_dotenv
load_dotenv()
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022B5336D2B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022B5336DFD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [16]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [17]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000022B5336D2B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000022B5336DFD0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out 

In [18]:
response=model_with_structure.invoke("Provide details about the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed structure

In [13]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movie rating out of 10")
    
model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception." Let me see. I need to use the Movie function they provided. The required parameters are title, year, director, and rating.\n\nFirst, I remember that "Inception" was directed by Christopher Nolan. The release year was 2010. The rating is a bit tricky, but I think it\'s around 8.8 on IMDb. Let me confirm those details. Yep, director is Nolan, year 2010, and the rating is 8.8. So I should structure the tool call with these parameters. Make sure all required fields are included. No need for optional parameters here. Alright, that should cover it.\n', 'tool_calls': [{'id': 'egjc5c4zc', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 196, 'prompt_tokens': 224, 'total_tokens': 420, 'completion_time': 0.33

### Nested Structure

In [14]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in million USD")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Arthur'), Actor(name='Tom Hardy', role='Eames')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [15]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year movie was released"]
    director: Annotated[str, ..., "the director of the movie"]
    rating: Annotated[float, ..., "the movie rating out of 10"]
    
model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avenger")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avenger', 'year': 2012}